In [0]:
SELECT *
FROM read_files('/Volumes/idplearn/default/idpproject')

In [0]:
CREATE OR REPLACE TABLE parsed_data AS
SELECT path,
ai_parse_document(content) as parsed_content
FROM read_files('/Volumes/idplearn/default/idpproject')


In [0]:
CREATE OR REPLACE TABLE pretty_data AS
SELECT path,
concat_ws('\n',
transform(try_cast(parsed_content:document:elements AS ARRAY<VARIANT>),
e -> coalesce(try_cast(e:content AS STRING),''))
) AS doc_text
FROM parsed_data

In [0]:
CREATE OR REPLACE TABLE classified_data AS
SELECT *,
ai_classify(doc_text, ARRAY('Invoice', 'Purchase Order', 'Receipt', 'Other')) AS doc_classification
FROM pretty_data

In [0]:
CREATE OR REPLACE TABLE invoice_data AS
SELECT *,
ai_extract(doc_text, ARRAY('vendor_name', 'Invoice_number', 'Invoice_Date', 'payment_method', 'total')) AS extracted
from classified_data
WHERE doc_classification = 'Invoice'

In [0]:
CREATE OR REPLACE TABLE idplearn.finance.invoices AS
SELECT path,
extracted.vendor_name AS vendor,
extracted.Invoice_number AS invoice_number,
extracted.Invoice_Date AS invoice_date,
extracted.payment_method AS payment_method,
extracted.total AS total
FROM invoice_data

In [0]:
CREATE SCHEMA IF NOT EXISTS idplearn.finance


In [0]:
SELECT *
FROM idplearn.finance.invoices

In [0]:
CREATE OR REPLACE TABLE Purchase_order_data AS
SELECT *,
ai_extract(doc_text, ARRAY('Merchant_Name', 'PO_Number','Purchase_Order_Date', 'Total')) AS extracted
from classified_data
WHERE doc_classification = 'Purchase Order'

In [0]:
CREATE OR REPLACE TABLE idplearn.finance.Purchase_order AS
SELECT path,
extracted.Merchant_Name AS Merchant_name,
extracted.PO_Number AS PO_Number,
extracted.Purchase_Order_Date AS Purchase_Order_Date,
extracted.Total AS Total
FROM Purchase_order_data

In [0]:
SELECT * FROM  idplearn.finance.Purchase_order

In [0]:
CREATE OR REPLACE TABLE Receipt_data AS
SELECT *,
ai_extract(doc_text, ARRAY('Merchant_Name', 'Receiept_Number','Transaction_Date', 'Total')) AS extracted
from classified_data
WHERE doc_classification = 'Receipt'

In [0]:
CREATE OR REPLACE TABLE idplearn.finance.Receipt AS
SELECT path,
extracted.Merchant_Name AS Merchant_name,
extracted.Receiept_Number AS Receiept_Number,
extracted.Transaction_Date AS Transaction_Date,
extracted.Total AS Total
FROM Receipt_data

In [0]:
select * from idplearn.finance.Receipt